# Lab 2 — Weighted Set Cover with Hill Climbing

**Goal:** Solve $S_1,\ldots,S_{100}$ using Hill Climbing.  
**State:** Boolean array (`True` selects a family).  
**Exploration improvements:** Randomized starts, three-bit exchanges, and multiple restarts.

## 1. Generate $S_n$

$U_n=\{0,\ldots,n-1\}$, $|f_i|=i+1$, and $c_i=10X_i+\sqrt{iY_i}$, where $X_i,Y_i\sim U(0,1)$.

In [25]:
from math import sqrt
from random import Random

def generate_instance(n, rng):
    universe = frozenset(range(n))
    families = tuple(
        frozenset(rng.sample(range(n), i + 1)) for i in range(n)
    )
    costs = tuple(
        10 * rng.random() + sqrt(i * rng.random()) for i in range(n)
    )
    return universe, families, costs

In [26]:
# Inspect a small instance
universe, families, costs = generate_instance(5, Random(42))
for i, family in enumerate(families):
    print(f"f{i}: {sorted(family)} | cost={costs[i]:.3f}")
assert families[-1] == universe
assert all(len(families[i]) == i + 1 for i in range(5))

f0: [0] | cost=5.054
f1: [0, 2] | cost=2.795
f2: [0, 1, 4] | cost=6.113
f3: [0, 1, 2, 4] | cost=7.451
f4: [0, 1, 2, 3, 4] | cost=1.860


## 2. Evaluate a Boolean solution

Maximize **fitness = −cost**. An incomplete cover has fitness $-\infty$.

In [27]:
def evaluate(solution, universe, families, costs):
    covered = set()
    total_cost = 0.0
    for selected, family, cost in zip(solution, families, costs):
        if selected:
            covered.update(family)
            total_cost += cost
    return total_cost, len(universe - covered)

def fitness(solution, universe, families, costs):
    cost, missing = evaluate(solution, universe, families, costs)
    return -cost if missing == 0 else float('-inf')

In [28]:
initial = [True] * len(families)
print("Initial fitness:", round(fitness(initial, universe, families, costs), 3))

Initial fitness: -23.273


## 3. Explore different feasible starting points

Randomly choose among the four families with the best **cost / newly covered objects** until all objects are covered.

In [29]:
def randomized_start(universe, families, costs, rng, top=4):
    solution = [False] * len(families)
    uncovered = set(universe)
    while uncovered:
        candidates = []
        for i, family in enumerate(families):
            if not solution[i]:
                gain = len(family & uncovered)
                if gain:
                    candidates.append((costs[i] / gain, i))
        candidates.sort()
        _, i = rng.choice(candidates[:top])
        solution[i] = True
        uncovered.difference_update(families[i])
    return solution

In [30]:
start = randomized_start(universe, families, costs, Random(7))
print("Initial selected families:", [i for i, chosen in enumerate(start) if chosen])
assert evaluate(start, universe, families, costs)[1] == 0

Initial selected families: [3, 4]


## 4. Define tweaks and feasibility checks

A tweak removes and/or adds selected families **in one move**. Coverage counts let us validate a candidate efficiently.

In [31]:
def tweak(solution, remove, add=()):
    candidate = solution.copy()
    for i in remove:
        candidate[i] = False
    for i in add:
        candidate[i] = True
    return candidate

def coverage_counts(solution, universe, families):
    counts = [0] * len(universe)
    for chosen, family in zip(solution, families):
        if chosen:
            for obj in family:
                counts[obj] += 1
    return counts

def feasible_move(counts, families, remove, add):
    affected = set()
    for i in remove:
        affected.update(families[i])
    for obj in affected:
        remaining = counts[obj] - sum(obj in families[i] for i in remove)
        if remaining == 0 and not any(obj in families[j] for j in add):
            return False
    return True

In [32]:
# An example of a three-bit exchange: one family out, two in.
example = [False, False, False, True]
neighbor = tweak(example, remove=(3,), add=(1, 2))
print("Before:", example)
print("After: ", neighbor)

Before: [False, False, False, True]
After:  [False, True, True, False]


## 5. Hill Climbing — exploitation and local exploration

- **Exploitation:** Accept the feasible move with the largest cost reduction.
- **Neighborhood:** All single removals and swaps, plus sampled three-bit exchanges.
- **Patience:** Stop after 3 consecutive rounds without improvement. Each unsuccessful round samples new exchange candidates.

In [33]:
def hill_climb(initial, universe, families, costs, rng, extra_samples=80, patience=3):
    if patience < 1:
        raise ValueError("patience must be at least 1")

    current = initial.copy()
    counts = coverage_counts(current, universe, families)
    assert all(counts)
    no_improvement = 0

    while True:
        selected = [i for i, flag in enumerate(current) if flag]
        unselected = [i for i, flag in enumerate(current) if not flag]
        best_move = None
        best_gain = 1e-10

        def consider(remove, add):
            nonlocal best_move, best_gain
            gain = sum(costs[i] for i in remove) - sum(costs[i] for i in add)
            if gain > best_gain and feasible_move(counts, families, remove, add):
                best_gain = gain
                best_move = (remove, add)

        # Exhaustive: one family removal or one-for-one exchange.
        for i in selected:
            consider((i,), ())
        for i in selected:
            for j in unselected:
                consider((i,), (j,))

        # Additional exploration: sampled three-bit exchanges.
        if len(unselected) >= 2:
            for _ in range(extra_samples):
                i = rng.choice(selected)
                j, k = rng.sample(unselected, 2)
                consider((i,), (j, k))
        if len(selected) >= 2 and unselected:
            for _ in range(extra_samples):
                i, j = rng.sample(selected, 2)
                k = rng.choice(unselected)
                consider((i, j), (k,))

        if best_move is None:
            no_improvement += 1
            if no_improvement >= patience:
                return current
            continue

        no_improvement = 0
        remove, add = best_move
        current = tweak(current, remove, add)
        for i in remove:
            for obj in families[i]:
                counts[obj] -= 1
        for j in add:
            for obj in families[j]:
                counts[obj] += 1

## 6. Multiple restarts — broader exploration

Run Hill Climbing from the all-selected state and randomized feasible states. Each run uses `patience=3`; keep the cheapest feasible result.

In [34]:
def solve(universe, families, costs, rng, restarts=6, extra_samples=80, patience=3):
    best = None
    best_cost = float('inf')
    for run in range(restarts):
        start = ([True] * len(families) if run == 0 else
                 randomized_start(universe, families, costs, rng))
        current = hill_climb(start, universe, families, costs, rng, extra_samples=extra_samples, patience=patience)
        cost, missing = evaluate(current, universe, families, costs)
        assert missing == 0
        if cost < best_cost:
            best, best_cost = current, cost
    return best

In [35]:
search_rng = Random(2026)
solution = solve(universe, families, costs, search_rng, restarts=6, extra_samples=80, patience=3)
total_cost, missing = evaluate(solution, universe, families, costs)
print("Selected families:", [i for i, chosen in enumerate(solution) if chosen])
print(f"Cost: {total_cost:.3f} | Uncovered objects: {missing}")
assert missing == 0

Selected families: [4]
Cost: 1.860 | Uncovered objects: 0


## 7. Solve the 100 required instances

Generate each $S_n$ once. Use independent random generators for the instances and the search.

In [36]:
instance_rng = Random(42)
search_rng = Random(2026)
results = []

for n in range(1, 101):
    universe, families, costs = generate_instance(n, instance_rng)

    assert len(families) == n
    assert all(len(families[i]) == i + 1 for i in range(n))
    assert families[-1] == universe

    solution = solve(
        universe, families, costs, search_rng,
        restarts=6, extra_samples=80, patience=3
    )
    total_cost, missing = evaluate(solution, universe, families, costs)
    assert missing == 0

    results.append({
        "n": n,
        "cost": total_cost,
        "selected_families": sum(solution),
        "uncovered": missing,
    })

print(f"Solved {len(results)} instances; all covers are feasible.")
for n in range(100):
    row = results[n - 1]
    print(f"S_{n}: cost={row['cost']:.3f}, families={row['selected_families']}")

Solved 100 instances; all covers are feasible.
S_0: cost=8.368, families=2
S_1: cost=0.250, families=1
S_2: cost=1.519, families=1
S_3: cost=4.100, families=1
S_4: cost=9.300, families=2
S_5: cost=6.919, families=1
S_6: cost=5.041, families=1
S_7: cost=4.724, families=1
S_8: cost=7.686, families=2
S_9: cost=3.769, families=1
S_10: cost=4.995, families=2
S_11: cost=7.045, families=1
S_12: cost=10.417, families=2
S_13: cost=5.746, families=1
S_14: cost=9.149, families=2
S_15: cost=9.516, families=1
S_16: cost=6.718, families=1
S_17: cost=8.718, families=1
S_18: cost=2.799, families=1
S_19: cost=6.609, families=2
S_20: cost=7.160, families=1
S_21: cost=9.198, families=1
S_22: cost=8.519, families=2
S_23: cost=7.747, families=2
S_24: cost=8.515, families=3
S_25: cost=4.704, families=1
S_26: cost=5.717, families=2
S_27: cost=6.117, families=1
S_28: cost=6.662, families=1
S_29: cost=10.034, families=1
S_30: cost=4.863, families=1
S_31: cost=7.850, families=3
S_32: cost=4.116, families=1
S_33